# Shift

Plots long-run average opinion vs. Bayesian bias intercept across topics and networks (Section 3.2). Reads from `outputs/shift/` and `outputs/bayesian/`.

In [1]:
import os
os.chdir("../")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
import json
import struct
import glob
from src import utils

def load_trajectory(filepath):
    with open(filepath, 'rb') as f:
        n_agents = struct.unpack('i', f.read(4))[0]
        n_timesteps = struct.unpack('i', f.read(4))[0]
        data = np.frombuffer(f.read(), dtype=np.float32)
        data = data.reshape((n_agents, n_timesteps), order='F')
    return data

sns.set_theme(context='paper', style='ticks', font_scale=1)

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

In [ ]:
name = "shift"
exp_name = "shift"
width_pt = 469

model_name = "google/gemma-3-12b-it"
quantification_method = "centroid"
keep_flips = False
bayesian_suffix = "__keep_flips" if keep_flips else ""

target_pct = 0.6
num_seeds = 20

model_str = model_name.replace("/", "_")
results_dir = "outputs/shift"


def result_path(network, dataset, task, transformed_pct, topic):
    return (
        f"{results_dir}/{exp_name}__network={network}"
        f"__transformed_pct={transformed_pct}"
        f"__dataset={dataset}__task={task}__topic={topic}"
        f"__model={model_str}__quantification_method={quantification_method}.json"
    )


def traj_path(network, dataset, task, transformed_pct, topic, seed):
    base = result_path(network, dataset, task, transformed_pct, topic)[:-len(".json")]
    return f"{base}_seed{seed}_trajectory.bin"


TOPIC_SHORTHAND = {
    "abortion": "ABO",
    "cloning": "CLO",
    "death_penalty": "DTP",
    "gun_control": "GNC",
    "marijuana_legalization": "MRJ",
    "minimum_wage": "MNW",
    "nuclear_energy": "NCL",
    "school_uniforms": "SCH",
    "atheism": "ATH",
    "acknowledging_climate_change": "CLI",
    "feminism": "FEM",
    "hillary_clinton": "HC",
    "donald_trump": "DT",
}


def topic_label(topic):
    return TOPIC_SHORTHAND.get(topic, topic)


def bayesian_intercept_mean(dataset, task, topic):
    path = (
        f"outputs/bayesian/{dataset}__{task}__{model_str}__{topic}"
        f"__{quantification_method}{bayesian_suffix}.json"
    )
    with open(path) as f:
        data = json.load(f)
    return data["model_direction"]["intercept"]["mean"]


def load_final_opinions(filepath):
    with open(filepath, "rb") as f:
        n_agents = struct.unpack("i", f.read(4))[0]
        n_timesteps = struct.unpack("i", f.read(4))[0]
        f.seek(8 + (n_timesteps - 1) * n_agents * 4)
        return np.frombuffer(f.read(n_agents * 4), dtype=np.float32).copy()

## Average equilibrium opinion vs Bayesian shift intercept

In [ ]:
shift_pattern = (
    f"{results_dir}/{exp_name}__network=*"
    f"__transformed_pct={target_pct}"
    f"__dataset=*__task=*__topic=*"
    f"__model={model_str}__quantification_method={quantification_method}.json"
)
network_combos = {}
for path in glob.glob(shift_pattern):
    basename = os.path.basename(path).replace(".json", "")
    parts = dict(p.split("=", 1) for p in basename.split("__")[1:])
    network_combos.setdefault(parts["network"], {}).setdefault(
        (parts["dataset"], parts["task"]), set()
    ).add(parts["topic"])
print(f"Found {len(network_combos)} networks")

palette = sns.color_palette("husl", 3)
DATASET_COLORS = {"semeval": palette[0], "ukp": palette[2]}
DATASET_MARKERS = {"semeval": "s", "ukp": "^"}
VANILLA_COLOR = palette[1]


def plot_opinion_vs_intercept(network, annotation_offsets=None,
                              default_offset=(12, 8)):
    by_ds = network_combos.get(network, {})
    if not by_ds:
        print(f"No data for network={network}")
        return
    annotation_offsets = annotation_offsets or {}

    vanilla_seed_means = []
    for (ds, tk), topics in by_ds.items():
        for tp in topics:
            if os.path.exists(traj_path(network, ds, tk, 0, tp, 1)):
                for seed in range(1, num_seeds + 1):
                    p = traj_path(network, ds, tk, 0, tp, seed)
                    if os.path.exists(p):
                        vanilla_seed_means.append(float(np.mean(load_final_opinions(p))))
                break
        if vanilla_seed_means:
            break
    vanilla_mean = float(np.mean(vanilla_seed_means)) if vanilla_seed_means else None

    points = []
    for (ds, tk), topics in by_ds.items():
        for tp in topics:
            seed_means = []
            for seed in range(1, num_seeds + 1):
                p = traj_path(network, ds, tk, target_pct, tp, seed)
                if os.path.exists(p):
                    seed_means.append(float(np.mean(load_final_opinions(p))))
            if not seed_means:
                continue
            try:
                intercept = bayesian_intercept_mean(ds, tk, tp)
            except FileNotFoundError:
                continue
            points.append((intercept, float(np.mean(seed_means)), ds, tp))

    if not points:
        return

    utils.latexify()
    fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    arrowprops = dict(arrowstyle="-", color="gray", lw=1)

    for intercept, opinion_avg, ds, tp in points:
        marker = DATASET_MARKERS.get(ds, "o")
        color = DATASET_COLORS.get(ds, "black")
        ax.scatter([intercept], [opinion_avg], marker=marker, color=color,
                   s=40, zorder=3)
        dx, dy = annotation_offsets.get((ds, tp), default_offset)
        ax.annotate(
            topic_label(tp),
            xy=(intercept, opinion_avg),
            xytext=(dx, dy),
            textcoords="offset points",
            ha="center", va="center",
            fontsize=8,
            color=color,
            arrowprops=arrowprops,
            zorder=5,
        )

    if vanilla_mean is not None:
        ax.scatter([0], [vanilla_mean], marker="x", color=VANILLA_COLOR,
                   s=60, linewidths=2, zorder=4)

    xs = np.array([p[0] for p in points])
    ys = np.array([p[1] for p in points])
    x_min_data = min(xs.min(), 0.0)
    x_max_data = max(xs.max(), 0.0)
    y_min_data = min(ys.min(), vanilla_mean) if vanilla_mean is not None else ys.min()
    y_max_data = max(ys.max(), vanilla_mean) if vanilla_mean is not None else ys.max()
    x_pad = (x_max_data - x_min_data) * 0.20 if x_max_data != x_min_data else 0.05
    y_pad = (y_max_data - y_min_data) * 0.20 if y_max_data != y_min_data else 0.02
    ax.set_xlim(x_min_data - x_pad, x_max_data + x_pad)
    ax.set_ylim(y_min_data - y_pad, y_max_data + y_pad)

    ax.axvline(x=0, color="gray", linestyle="--", linewidth=1, zorder=0)
    ax.axhline(y=vanilla_mean, color="gray", linestyle="--", linewidth=1, zorder=0)

    if len(points) >= 2:
        r, p_value = stats.pearsonr(xs, ys)
        if p_value < 0.05:
            p_format = "< 0.05"
        else:
            p_format = f"= {float(np.round(p_value, 3)):.2f}"
        ax.text(
            0.05, 0.95, f"$r = {r:.2f}$\n$p {p_format}$",
            transform=ax.transAxes, va="top", ha="left", fontsize=8,
        )

    handles = [
        Line2D([0], [0], marker=DATASET_MARKERS["semeval"], linestyle="",
               color=DATASET_COLORS["semeval"], markersize=7, label="SemEval"),
        Line2D([0], [0], marker=DATASET_MARKERS["ukp"], linestyle="",
               color=DATASET_COLORS["ukp"], markersize=7, label="UKP"),
        Line2D([0], [0], marker="x", linestyle="",
               color=VANILLA_COLOR, markersize=8, markeredgewidth=2, label="None"),
    ]
    ax.legend(handles=handles, title="AI transformation",
              loc="lower right", framealpha=0.9, fontsize=8, title_fontsize=8)

    sns.despine(ax=ax)
    ax.set_xlabel(r"Bias towards ``in favor''")
    ax.set_ylabel("Long-run average opinion")

    fig.tight_layout()
    fig.savefig(
        f"figures/{name}__opinion_vs_intercept__{model_str}"
        f"__network={network}__target_pct={target_pct}.pdf",
        dpi=300,
    )
    plt.close()
    print(f"Saved figure for network={network} ({len(points)} topics)")

In [ ]:
plot_opinion_vs_intercept(
    "twitter",
    annotation_offsets={
        ("semeval", "atheism"):                       (-22, -10),
        ("semeval", "donald_trump"):                  (-18, 13),
        ("semeval", "hillary_clinton"):               (-22, 12),
        ("semeval", "abortion"):                      (-20, 12),
        ("semeval", "acknowledging_climate_change"):  (24, 12),
        ("semeval", "feminism"):                      (16, 14),
        ("ukp", "death_penalty"):                     (-15, -13),
        ("ukp", "school_uniforms"):                   (-8, -16),
        ("ukp", "cloning"):                           (10, -16),
        ("ukp", "minimum_wage"):                      (-22, 14),
        ("ukp", "marijuana_legalization"):            (-5, 16),
        ("ukp", "nuclear_energy"):                    (25, 12),
        ("ukp", "abortion"):                          (-20, -12),
        ("ukp", "gun_control"):                       (4, -15),
    },
)

In [ ]:
plot_opinion_vs_intercept(
    "facebook",
    annotation_offsets={
        ("semeval", "atheism"):                       (-22, -10),
        ("semeval", "donald_trump"):                  (19, -5),
        ("semeval", "hillary_clinton"):               (-22, 14),
        ("semeval", "abortion"):                      (14, -14),
        ("semeval", "acknowledging_climate_change"):  (20, 12),
        ("semeval", "feminism"):                      (20, -12),
        ("ukp", "death_penalty"):                     (17, -13),
        ("ukp", "school_uniforms"):                   (-8, 15),
        ("ukp", "cloning"):                           (-15, -14),
        ("ukp", "minimum_wage"):                      (-22, 14),
        ("ukp", "marijuana_legalization"):            (-4, 16),
        ("ukp", "nuclear_energy"):                    (20, 8),
        ("ukp", "abortion"):                          (-4, -16),
        ("ukp", "gun_control"):                       (18, 10),
    },
)

In [ ]:
plot_opinion_vs_intercept(
    "gplus",
    annotation_offsets={
        ("semeval", "atheism"):                       (-22, -10),
        ("semeval", "donald_trump"):                  (-15, -14),
        ("semeval", "hillary_clinton"):               (-22, 0),
        ("semeval", "abortion"):                      (-22, 14),
        ("semeval", "acknowledging_climate_change"):  (-22, 12),
        ("semeval", "feminism"):                      (-20, 14),
        ("ukp", "death_penalty"):                     (-12, -14),
        ("ukp", "school_uniforms"):                   (0, -16),
        ("ukp", "cloning"):                           (12, -14),
        ("ukp", "minimum_wage"):                      (4, -14),
        ("ukp", "marijuana_legalization"):            (8, 18),
        ("ukp", "nuclear_energy"):                    (24, -4),
        ("ukp", "abortion"):                          (-14, 14),
        ("ukp", "gun_control"):                       (24, 4),
    },
)